# Xenium morphology + cells in napari

Xenium morphology image → cell centers / clusters → cell boundaries → napari

In [1]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ubuntu + Wayland
os.environ["QT_QPA_PLATFORM"] = "xcb"
os.environ["PYOPENGL_PLATFORM"] = "glx"

import napari

In [3]:
PROJECT = Path.home() / "Projects/xenium_lung"

VIS_DIR = PROJECT / "results/xenium_5k/visualization"
RAW_DIR = PROJECT / "data/bundle/xenium_prime_5k"

COORDINATES_FILE = VIS_DIR / "cell_coordinates.parquet"
CLUSTERS_FILE = VIS_DIR / "cluster_labels.csv"
BOUNDARIES_FILE = VIS_DIR / "cell_boundaries.parquet"

MORPHOLOGY_FILE = RAW_DIR / "morphology.ome.tif"

print("Morphology:", MORPHOLOGY_FILE)
print("Exists:", MORPHOLOGY_FILE.exists())

Morphology: /home/duydao/Projects/xenium_lung/data/bundle/xenium_prime_5k/morphology.ome.tif
Exists: True


# Load the cell information

In [4]:
coordinates = pd.read_parquet(COORDINATES_FILE)
clusters = pd.read_csv(CLUSTERS_FILE)

cells = coordinates.merge(
    clusters,
    on="cell_id",
    how="left",
    validate="one_to_one"
)

print("Cells:", cells.shape)
print(cells.head())

Cells: (266179, 4)
      cell_id           x            y  group
0  aaaaadnb-1  822.469055  5111.537598      0
1  aaaabalp-1  843.901428  5149.261230      0
2  aaaadfei-1  831.219421  5133.179199      1
3  aaaadjia-1  839.742981  5159.693848      0
4  aaaafglb-1  784.250916  5143.785645      1


In [5]:
cluster_ids = sorted(
    cells["group"].dropna().unique()
)

cmap = plt.get_cmap(
    "tab20",
    len(cluster_ids)
)

cluster_colors = {
    cluster_id: cmap(i)
    for i, cluster_id in enumerate(cluster_ids)
}

point_colors = np.array([
    cluster_colors[group]
    if not pd.isna(group)
    else (0.5, 0.5, 0.5, 1.0)
    for group in cells["group"]
])

print("Clusters:", cluster_ids)

Clusters: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]


# Inspect the morphology TIFF

Before opening it in napari, let's inspect its dimensions.

In [6]:
import tifffile

with tifffile.TiffFile(MORPHOLOGY_FILE) as tif:
    print(tif)

    print("\nNumber of pages:", len(tif.pages))

    for i, page in enumerate(tif.pages[:5]):
        print(
            i,
            "shape =", page.shape,
            "dtype =", page.dtype
        )

TiffFile 'morphology.ome.tif'  9.65 GiB  BigTiff  17 Pages  ome

Number of pages: 17
0 shape = (37348, 54086) dtype = uint16
1 shape = (37348, 54086) dtype = uint16
2 shape = (37348, 54086) dtype = uint16
3 shape = (37348, 54086) dtype = uint16
4 shape = (37348, 54086) dtype = uint16


In [7]:
with tifffile.TiffFile(MORPHOLOGY_FILE) as tif:
    ome_xml = tif.ome_metadata

print(ome_xml[:3000])

<OME xmlns="http://www.openmicroscopy.org/Schemas/OME/2016-06" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.openmicroscopy.org/Schemas/OME/2016-06 http://www.openmicroscopy.org/Schemas/OME/2016-06/ome.xsd">
  <Plate ID="Plate:0" WellOriginX="0.0" WellOriginXUnit="µm" WellOriginY="0.0" WellOriginYUnit="µm"/>
  <Instrument ID="Instrument:0">
    <Microscope Manufacturer="10x Genomics" Model="Xenium"/>
  </Instrument>
  <Image ID="Image:0">
    <InstrumentRef ID="Instrument:0"/>
    <Pixels ID="Pixels:0" DimensionOrder="XYZCT" Type="uint16" SizeX="54086" SizeY="37348" SizeZ="17" SizeC="1" SizeT="1" PhysicalSizeX="0.2125" PhysicalSizeXUnit="µm" PhysicalSizeY="0.2125" PhysicalSizeYUnit="µm" PhysicalSizeZ="3.0" PhysicalSizeZUnit="µm">
      <Channel ID="Channel:0" Name="DAPI" SamplesPerPixel="1">
        <AnnotationRef ID="Annotation:0"/>
      </Channel>
      <TiffData PlaneCount="17"/>
    </Pixels>
    <AnnotationRef ID="Annotation:0"/>
  </Image>


In [21]:
import re

# cell x/y are in microns; the TIFF pixel grid needs this scale to line up with them
match_x = re.search(r'PhysicalSizeX="([\d.]+)"', ome_xml)
match_y = re.search(r'PhysicalSizeY="([\d.]+)"', ome_xml)

pixel_size_x = float(match_x.group(1)) if match_x else 1.0
pixel_size_y = float(match_y.group(1)) if match_y else 1.0

print("Pixel size (x, y):", pixel_size_x, pixel_size_y)

Pixel size (x, y): 0.2125 0.2125


# Open napari

In [22]:
viewer = napari.Viewer()

# Load morphology image

In [23]:
# lazy-load via zarr store — avoids reading the full 64 GiB array into RAM
import zarr

store = tifffile.imread(MORPHOLOGY_FILE, aszarr=True)
morphology_zarr = zarr.open(store, mode="r")

print(morphology_zarr.info if hasattr(morphology_zarr, "info") else morphology_zarr)

Name        : 
Type        : Group
Zarr format : 2
Read-only   : True
Store type  : ZarrTiffStore


# Alternative: explicitly load with tifffile

The morphology TIFF is a multiscale (pyramidal) OME-TIFF — `tifffile.imread` on the full series requires 64 GiB of RAM. Instead build a list of dask arrays, one per resolution level, and hand that pyramid to napari so only visible tiles are read from disk.

In [24]:
import dask.array as da
import zarr


def _open_level_array(level):
    node = zarr.open(level.aszarr(), mode="r")
    if isinstance(node, zarr.Array):
        return node
    # multi-array group — the dataset for this level is the first array member
    key = next(iter(node.array_keys()))
    return node[key]


with tifffile.TiffFile(MORPHOLOGY_FILE) as tif:
    series = tif.series[0]
    levels = series.levels

    pyramid = [
        da.from_array(_open_level_array(level), chunks="auto")
        for level in levels
    ]

for i, level in enumerate(pyramid):
    print(i, "shape =", level.shape, "dtype =", level.dtype)

0 shape = (17, 37348, 54086) dtype = uint16
1 shape = (17, 18674, 27043) dtype = uint16
2 shape = (17, 9337, 13521) dtype = uint16
3 shape = (17, 4668, 6760) dtype = uint16
4 shape = (17, 2334, 3380) dtype = uint16
5 shape = (17, 1167, 1690) dtype = uint16
6 shape = (17, 583, 845) dtype = uint16
7 shape = (17, 291, 422) dtype = uint16


In [25]:
viewer.add_image(
    pyramid,
    name="Morphology",
    multiscale=True,
    contrast_limits=[0, 20000],
    scale=(pixel_size_y, pixel_size_x),
)

<Image layer 'Morphology' at 0x772536963a90>

# Add cells

Assuming the morphology image is successfully loaded.

In [26]:
all_points = cells[["y", "x"]].to_numpy(
    dtype=np.float32
)

viewer.add_points(
    all_points,
    size=2,
    face_color=point_colors,
    name="Cells — clusters",
)

<Points layer 'Cells — clusters' at 0x7724eab3fa50>

# Add boundaries

Not adding all 266k boundaries yet — reuse the boundary subset function from notebook 04.

In [27]:
from shapely import from_wkb

def add_boundary_subset(
    viewer,
    boundaries,
    cluster_lookup,
    cluster_colors,
    n_polygons=5000,
    layer_name=None,
):

    subset = boundaries.iloc[:n_polygons]

    geometries = from_wkb(
        subset["geometry"].to_numpy()
    )

    polygon_data = []
    polygon_groups = []

    for cell_id, geom in zip(
        subset.index,
        geometries
    ):

        group = cluster_lookup.get(
            str(cell_id),
            np.nan
        )

        if geom.geom_type == "Polygon":

            coords = np.asarray(
                geom.exterior.coords
            )

            # Shapely X,Y → napari Y,X
            coords = coords[:, [1, 0]]

            polygon_data.append(coords)
            polygon_groups.append(group)

        elif geom.geom_type == "MultiPolygon":

            for polygon in geom.geoms:

                coords = np.asarray(
                    polygon.exterior.coords
                )

                coords = coords[:, [1, 0]]

                polygon_data.append(coords)
                polygon_groups.append(group)

    edge_colors = np.array([
        cluster_colors[group]
        if not pd.isna(group)
        else (0.5, 0.5, 0.5, 1.0)
        for group in polygon_groups
    ])

    if layer_name is None:
        layer_name = (
            f"Cell boundaries — {n_polygons:,}"
        )

    return viewer.add_shapes(
        polygon_data,
        shape_type="polygon",
        edge_width=0.8,
        edge_color=edge_colors,
        face_color=[0, 0, 0, 0],
        name=layer_name,
    )

In [28]:
boundaries = pd.read_parquet(
    BOUNDARIES_FILE
)

cluster_lookup = cells.set_index(
    "cell_id"
)["group"]

boundary_layer = add_boundary_subset(
    viewer,
    boundaries,
    cluster_lookup,
    cluster_colors,
    n_polygons=5_000,
)

# Check coordinate alignment

This is the most important part of this notebook. Cell coordinates are in Xenium physical `x`/`y`, while the TIFF has its own pixel coordinate system. If the morphology image and cell locations don't line up, don't manually flip axes or guess a scale factor yet — first inspect the image metadata/alignment information (`Xenium_Prime_Human_Lung_Cancer_FFPE_he_imagealignment.csv`).

In [29]:
alignment_files = list(
    RAW_DIR.glob("*imagealignment*.csv")
)

for f in alignment_files:
    print(f)

/home/duydao/Projects/xenium_lung/data/bundle/xenium_prime_5k/Xenium_Prime_Human_Lung_Cancer_FFPE_he_imagealignment.csv


In [30]:
ALIGNMENT_FILE = alignment_files[0]

alignment = pd.read_csv(
    ALIGNMENT_FILE
)

print(alignment)
print("\nColumns:")
print(alignment.columns.tolist())

   0.009888992716531058  1.2882591434922286  -755.7872298494876
0             -1.288259            0.009889        34502.597474
1              0.000000            0.000000            1.000000

Columns:
['0.009888992716531058', '1.2882591434922286', '-755.7872298494876']


# Very important distinction

The raw Xenium bundle contains at least two relevant images: `morphology.ome.tif` (Xenium morphology) and `Xenium_Prime_Human_Lung_Cancer_FFPE_he_image.ome.tif` (H&E). Eventually create two separate napari image layers:

```
H&E
 ↓
Morphology
 ↓
Cell boundaries
 ↓
Cell clusters
 ↓
Transcripts
```

Useful for identifying tumor regions, stromal regions, fibroblast/CAF-rich regions, immune infiltrates, tumor–stroma interfaces, and lymphoid aggregates — fits the NSCLC TME analysis well.

**Caution:** don't assume `morphology.ome.tif` and the H&E TIFF share the same coordinate system or dimensions. The alignment CSV exists precisely because an image-to-Xenium coordinate transformation may be required.

After running the TIFF metadata cell and the alignment-read cell, check `morphology.shape`, `alignment`, and `alignment.columns.tolist()` before attempting any axis flip or scale-factor alignment.